# 9x9 Single Prompt Evaluation

Run puzzle 06 and 07 with four Gemini models and save one result JSON per model/puzzle.

In [4]:
%pip install google-genai
%pip install pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import json
import os
import time
from datetime import datetime
from pathlib import Path
from typing import List

from google import genai
from google.genai import types
from pydantic import BaseModel, Field

os.environ['GOOGLE_CLOUD_PROJECT'] = 'project-038ccd57-3d62-4aac-8b5'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

client = genai.Client()

cwd = Path.cwd()
if (cwd / 'killer_sudoku_cheat_sheet_9x9.md').exists():
    NINE_BY_NINE_DIR = cwd
elif (cwd / '9x9' / 'killer_sudoku_cheat_sheet_9x9.md').exists():
    NINE_BY_NINE_DIR = cwd / '9x9'
elif (cwd / 'cs106' / '9x9' / 'killer_sudoku_cheat_sheet_9x9.md').exists():
    NINE_BY_NINE_DIR = cwd / 'cs106' / '9x9'
else:
    raise FileNotFoundError('Cannot locate cs106/9x9 resources from current working directory.')

CS106_DIR = NINE_BY_NINE_DIR.parent
DATASET_DIR = CS106_DIR / 'dataset'
OUTPUT_ROOT = CS106_DIR / 'outputs' / '9x9'

MODELS_LLM = [
    'gemini-2.5-flash',
    'gemini-2.5-pro',
    'gemini-3.5-flash',
    'gemini-3.1-flash-lite',
]
PUZZLE_FILES = [DATASET_DIR / f'puzzle_{i:02d}.json' for i in (6, 7)]


REQUEST_DELAY_SECONDS = 5
STOP_ON_QUOTA = True

print('9x9 dir:', NINE_BY_NINE_DIR)
print('Dataset files:', [str(p) for p in PUZZLE_FILES])
print('Output root:', OUTPUT_ROOT)

9x9 dir: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\9x9
Dataset files: ['d:\\Study\\HK6\\CS106-AI\\DoAn\\CS106-Sudoku-Bench\\cs106\\dataset\\puzzle_06.json', 'd:\\Study\\HK6\\CS106-AI\\DoAn\\CS106-Sudoku-Bench\\cs106\\dataset\\puzzle_07.json']
Output root: d:\Study\HK6\CS106-AI\DoAn\CS106-Sudoku-Bench\cs106\outputs\9x9


In [6]:
def grid_to_markdown(grid):
    rows = []
    rows.append('| ' + ' | '.join([''] * len(grid[0])) + ' |')
    rows.append('|' + '|'.join(['---'] * len(grid[0])) + '|')
    for row in grid:
        rows.append('| ' + ' | '.join(str(cell) for cell in row) + ' |')
    return '\n'.join(rows)


def box_shape_for(grid_size):
    if grid_size == 9:
        return '3x3', 3, 3
    if grid_size == 6:
        return '2x3', 2, 3
    raise ValueError(f'Unsupported grid size: {grid_size}')


def build_cages_text(puzzle):
    cages_text = []
    for cage in puzzle['cages']:
        cell_strs = [f'r{r+1}c{c+1}' for r, c in cage['cells']]
        cages_text.append('- ' + ' + '.join(cell_strs) + f" = {cage['sum']}")
    return cages_text


def model_output_parts(model):
    model_lower = model.lower()
    if '2.5' in model_lower:
        family = 'gemini-2.5'
    elif '3.' in model_lower:
        family = 'gemini-3.x'
    else:
        family = 'other'

    if 'lite' in model_lower:
        variant = 'lite'
    elif 'pro' in model_lower:
        variant = 'pro'
    elif 'flash' in model_lower:
        variant = 'flash'
    else:
        variant = 'other'

    return family, variant


with open(NINE_BY_NINE_DIR / 'killer_sudoku_cheat_sheet_9x9.md', 'r', encoding='utf-8') as file:
    cheat_sheet = file.read()



In [7]:
prompt = """
You are an expert Killer Sudoku solver.

Your task is to solve the Killer Sudoku puzzle based on the rules, reference material, and puzzle data provided below.

1. Grid & Core Rules

- The grid size is {grid_size}x{grid_size}.
- Notation: "r1c1" means row 1, column 1.
- Each cell must contain one digit from 1 to {grid_size}.
- Subgrid shape: {box_shape}.

Standard Sudoku Rules:
- Each row must contain digits 1 to {grid_size} exactly once.
- Each column must contain digits 1 to {grid_size} exactly once.
- Each subgrid must contain digits 1 to {grid_size} exactly once.

Killer Sudoku Rules:
- The grid is divided into cages.
- The digits in each cage must sum to the cage target.
- Digits cannot repeat within the same cage.

2. Mathematical Facts

For this puzzle:
- Row sum: {unit_sum}
- Column sum: {unit_sum}
- Subgrid sum: {unit_sum}
- Number of cage equations: {cage_count}
- Total sum equations: {total_equations}

3. Reference Material: Killer Sudoku Cheat Sheet

- Combinations are written as digit sequences without separators.
- Example: "12" means {{1, 2}}
- Example: "135" means {{1, 3, 5}}

Cheat sheet:

{cheat_sheet}

4. Puzzle Instance

Initial State:

{puzzle}

Cages:

{cages}

5. Solving Requirements

Solve the puzzle logically using:
- Sudoku row constraints
- Sudoku column constraints
- Sudoku subgrid constraints
- Killer cage sum constraints
- Cage distinctness constraints
- Cheat sheet combinations

Do NOT guess.
Do NOT invent values.
Only return a solution if it is logically consistent with ALL constraints.

6. Mandatory Self-Verification Before Output

Before returning the final solution, silently verify ALL of the following:

Sudoku validation:
- Every row contains digits 1 to {grid_size} exactly once.
- Every column contains digits 1 to {grid_size} exactly once.
- Every subgrid contains digits 1 to {grid_size} exactly once.

Killer Sudoku validation:
- Every cage sums exactly to its target.
- No repeated digit exists inside any cage.

Global validation:
- Every cell is filled.
- No contradictions exist.

If ANY validation fails:
DO NOT return an invalid board.

7. Output Requirement

Return ONLY valid JSON matching this schema:

{{
  "board": [[...]]
}}

If you cannot determine a fully valid solution with certainty, return:

{{
  "board": []
}}

Do not include:
- explanations
- markdown
- comments
- reasoning text
- extra text
"""

In [8]:
class SudokuResult(BaseModel):
    board: List[List[int]] = Field(
        description="The solved Sudoku grid as a 2D list of integers."
    )

In [9]:
def normalize_board(board):
    if board is None:
        return None
    if board == []:
        return []
    return [[int(cell) for cell in row] for row in board]


def validate_prediction(board, puzzle):
    errors = []
    grid_size = puzzle['grid_size']
    expected = list(range(1, grid_size + 1))
    _, box_rows, box_cols = box_shape_for(grid_size)

    if board is None:
        return False, 'No Prediction', ['prediction is None']
    if board == []:
        return False, 'Empty Prediction', ['model returned an empty board']
    if len(board) != grid_size or any(len(row) != grid_size for row in board):
        return False, 'Invalid Shape', [f'expected {grid_size}x{grid_size}, got {len(board)} rows']

    for i, row in enumerate(board):
        if sorted(row) != expected:
            errors.append(f'Row {i + 1} invalid: {row}')

    for col in range(grid_size):
        column = [board[row][col] for row in range(grid_size)]
        if sorted(column) != expected:
            errors.append(f'Column {col + 1} invalid: {column}')

    for row_start in range(0, grid_size, box_rows):
        for col_start in range(0, grid_size, box_cols):
            nums = []
            for r in range(row_start, row_start + box_rows):
                for c in range(col_start, col_start + box_cols):
                    nums.append(board[r][c])
            if sorted(nums) != expected:
                errors.append(f'Subgrid ({row_start + 1},{col_start + 1}) invalid: {nums}')

    for idx, cage in enumerate(puzzle['cages']):
        values = [board[r][c] for r, c in cage['cells']]
        if sum(values) != cage['sum']:
            errors.append(f"Cage {idx + 1} sum invalid: got {sum(values)}, expected {cage['sum']}, values={values}")
        if len(values) != len(set(values)):
            errors.append(f'Cage {idx + 1} repeated digits: values={values}')

    if board != puzzle['solution']:
        errors.append('Prediction differs from provided solution')

    if errors:
        if any(err.startswith('Prediction differs') for err in errors) and len(errors) == 1:
            return False, 'Different From Solution', errors
        return False, 'Constraint Violation', errors

    return True, 'None', []


def build_config(model):
    if model == "gemini-3.1-flash-lite":
        return types.GenerateContentConfig(
            temperature=0,
            response_json_schema=SudokuResult.model_json_schema(),
            response_mime_type="application/json",
            thinking_config=types.ThinkingConfig(
                thinking_level=types.ThinkingLevel.MEDIUM
            )
        )

    return types.GenerateContentConfig(
        response_json_schema=SudokuResult.model_json_schema(),
        response_mime_type="application/json"
    )


def run_single_prompt(model, puzzle_file):
    with open(puzzle_file, 'r', encoding='utf-8') as f:
        puzzle = json.load(f)

    grid_size = puzzle['grid_size']
    box_shape, _, _ = box_shape_for(grid_size)
    unit_sum = grid_size * (grid_size + 1) // 2
    cage_count = len(puzzle['cages'])
    total_equations = grid_size * 3 + cage_count
    cages_text = build_cages_text(puzzle)
    markdown_table = grid_to_markdown(puzzle['puzzle'])

    start_time = time.time()
    result = None
    raw_response = None
    prediction = None
    error_message = None
    is_correct = False
    error_type = 'None'
    validation_errors = []

    try:
        response = client.models.generate_content(
            model=model,
            contents=prompt.format(
                grid_size=grid_size,
                box_shape=box_shape,
                unit_sum=unit_sum,
                cage_count=cage_count,
                total_equations=total_equations,
                puzzle=markdown_table,
                cages='\n'.join(cages_text),
                cheat_sheet=cheat_sheet,
            ),
            config=build_config(model),
        )
        raw_response = response.text
        result = SudokuResult.model_validate_json(response.text)
        prediction = normalize_board(result.board)
        is_correct, error_type, validation_errors = validate_prediction(prediction, puzzle)
    except Exception as e:
        error_message = str(e)
        if '429' in error_message or 'RESOURCE_EXHAUSTED' in error_message:
            error_type = 'Resource Exhausted'
        else:
            error_type = type(e).__name__
        validation_errors = [error_message]

    elapsed = time.time() - start_time
    family, variant = model_output_parts(model)
    output_dir = OUTPUT_ROOT / family / 'single-prompt' / variant
    output_dir.mkdir(parents=True, exist_ok=True)
    out_file = output_dir / f"result_puzzle_{puzzle['id']:02d}.json"

    log_data = {
        'puzzle_id': puzzle['id'],
        'difficulty': puzzle['difficulty'],
        'grid_size': puzzle['grid_size'],
        'model': model,
        'status': 'Success' if is_correct else 'Failed',
        'error_type': error_type,
        'error_message': error_message,
        'validation_errors': validation_errors,
        'time_seconds': elapsed,
        'prediction': prediction,
        'solution': puzzle['solution'],
        'raw_response': raw_response if result is None else None,
        'output_file': str(out_file),
        'finished_at': datetime.now().isoformat(timespec='seconds'),
    }

    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, ensure_ascii=False, indent=2)

    return log_data

In [ ]:
all_results = []
quota_exhausted = False

for model in MODELS_LLM:
    if quota_exhausted and STOP_ON_QUOTA:
        break

    print(f'\nEvaluating model: {model}')
    for puzzle_file in PUZZLE_FILES:
        print(f'  Running {puzzle_file.name}...')
        log_data = run_single_prompt(model, puzzle_file)
        all_results.append(log_data)

        print(
            f"    Status: {log_data['status']} | "
            f"Error: {log_data['error_type']} | "
            f"Time: {log_data['time_seconds']:.2f}s"
        )
        print(f"    Saved: {log_data['output_file']}")

        if log_data['error_type'] == 'Resource Exhausted' and STOP_ON_QUOTA:
            quota_exhausted = True
            print('    Quota exhausted. Stopping remaining requests.')
            break

        time.sleep(REQUEST_DELAY_SECONDS)

print('\nSummary')
for item in all_results:
    print(
        f"Puzzle {item['puzzle_id']:02d} | {item['model']} | "
        f"{item['status']} | {item['error_type']} | {item['time_seconds']:.2f}s"
    )


Evaluating model: gemini-2.5-flash
  Running puzzle_06.json...
